In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score

In [2]:
# Load AWGN dataset and extracted features
awgn_data = np.load("spectrum_sensing_dataset.npz")
awgn_features = np.load("spectrum_sensing_features.npz")

X_iq = awgn_data["X"]
y = awgn_data["y"]
snr = awgn_data["snr"]

X_features = awgn_features["X"]
feature_names = awgn_features["feature_names"]

# Load Rayleigh dataset and extracted features
rayleigh_data = np.load("rayleigh_fading_dataset.npz")
rayleigh_features = np.load("rayleigh_fading_features.npz")

X_rayleigh_iq = rayleigh_data["X"]
X_rayleigh_features = rayleigh_features["X"]

print("AWGN IQ shape:", X_iq.shape)
print("AWGN feature shape:", X_features.shape)
print("Rayleigh IQ shape:", X_rayleigh_iq.shape)
print("Rayleigh feature shape:", X_rayleigh_features.shape)

print("Labels:", np.unique(y))
print("SNR values:", np.unique(snr))
print("Feature count:", len(feature_names))

AWGN IQ shape: (14000, 256)
AWGN feature shape: (14000, 13)
Rayleigh IQ shape: (14000, 256)
Rayleigh feature shape: (14000, 13)
Labels: [0 1]
SNR values: [-20 -15 -10  -5   0   5  10]
Feature count: 13


In [3]:
# Random seeds for repeatability experiments
random_seeds = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

# Experiment settings
TEST_SIZE = 0.30
VALIDATION_SIZE = 0.50  # Half of the remaining 30% = 15% overall

# Combined label-SNR groups for stratification
stratification_groups = np.array([
    f"{label}_{snr_value}"
    for label, snr_value in zip(y, snr)
])

# Verify the distribution of samples
print("Number of random seeds:", len(random_seeds))
print("Total samples:", len(y))
print("\nSamples per label:")
print(pd.Series(y).value_counts().sort_index())

print("\nSamples per label-SNR combination:")
print(pd.crosstab(snr, y))

Number of random seeds: 10
Total samples: 14000

Samples per label:
0    7000
1    7000
Name: count, dtype: int64

Samples per label-SNR combination:
col_0     0     1
row_0            
-20    1000  1000
-15    1000  1000
-10    1000  1000
-5     1000  1000
 0     1000  1000
 5     1000  1000
 10    1000  1000


In [4]:
def create_data_split(y, snr, random_state):

    indices = np.arange(len(y))

    groups = np.array([
        f"{label}_{snr_value}"
        for label, snr_value in zip(y, snr)
    ])

    # First split: 70% training, 30% temporary
    train_idx, temp_idx = train_test_split(
        indices,
        test_size=0.30,
        random_state=random_state,
        stratify=groups
    )

    # Second split: 15% validation, 15% testing
    temp_groups = groups[temp_idx]

    val_idx, test_idx = train_test_split(
        temp_idx,
        test_size=0.50,
        random_state=random_state,
        stratify=temp_groups
    )

    return train_idx, val_idx, test_idx

In [5]:
train_idx, val_idx, test_idx = create_data_split(
    y, snr, random_seeds[0]
)

print("Training samples:", len(train_idx))
print("Validation samples:", len(val_idx))
print("Testing samples:", len(test_idx))

print("\nTotal samples:", len(train_idx) + len(val_idx) + len(test_idx))

print("\nTraining class distribution:")
print(pd.Series(y[train_idx]).value_counts().sort_index())

print("\nValidation class distribution:")
print(pd.Series(y[val_idx]).value_counts().sort_index())

print("\nTesting class distribution:")
print(pd.Series(y[test_idx]).value_counts().sort_index())

Training samples: 9800
Validation samples: 2100
Testing samples: 2100

Total samples: 14000

Training class distribution:
0    4900
1    4900
Name: count, dtype: int64

Validation class distribution:
0    1050
1    1050
Name: count, dtype: int64

Testing class distribution:
0    1050
1    1050
Name: count, dtype: int64


In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    roc_auc_score
)

In [7]:
def calculate_metrics(y_true, y_prob, threshold):

    y_pred = (y_prob > threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    accuracy = accuracy_score(y_true, y_pred)
    pd_value = tp / (tp + fn)
    pfa_value = fp / (fp + tn)
    pm_value = fn / (fn + tp)
    auc_value = roc_auc_score(y_true, y_prob)

    return {
        "Accuracy": accuracy,
        "Pd": pd_value,
        "Pfa": pfa_value,
        "Pm": pm_value,
        "ROC_AUC": auc_value,
        "Threshold": threshold
    }

In [8]:
results = []

for seed in random_seeds:

    print(f"Running experiment with seed {seed}...")

    # Generate split
    train_idx, val_idx, test_idx = create_data_split(
        y, snr, seed
    )

    # Split features and labels
    X_train = X_features[train_idx]
    X_val = X_features[val_idx]
    X_test = X_features[test_idx]

    y_train = y[train_idx]
    y_val = y[val_idx]
    y_test = y[test_idx]

    # --------------------------------
    # 1. LOGISTIC REGRESSION
    # --------------------------------

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    lr_model = LogisticRegression(
        max_iter=1000,
        random_state=seed,
        class_weight="balanced"
    )

    lr_model.fit(X_train_scaled, y_train)

    lr_val_prob = lr_model.predict_proba(X_val_scaled)[:, 1]
    lr_test_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

    lr_threshold = np.quantile(
        lr_val_prob[y_val == 0],
        0.95,
        method="higher"
    )

    lr_metrics = calculate_metrics(
        y_test, lr_test_prob, lr_threshold
    )

    results.append({
        "Seed": seed,
        "Channel": "AWGN",
        "Model": "Logistic Regression",
        **lr_metrics
    })

    # --------------------------------
    # 2. RANDOM FOREST
    # --------------------------------

    rf_model = RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=seed,
        n_jobs=-1,
        class_weight="balanced"
    )

    rf_model.fit(X_train, y_train)

    rf_val_prob = rf_model.predict_proba(X_val)[:, 1]
    rf_test_prob = rf_model.predict_proba(X_test)[:, 1]

    rf_threshold = np.quantile(
        rf_val_prob[y_val == 0],
        0.95,
        method="higher"
    )

    rf_metrics = calculate_metrics(
        y_test, rf_test_prob, rf_threshold
    )

    results.append({
        "Seed": seed,
        "Channel": "AWGN",
        "Model": "Random Forest",
        **rf_metrics
    })

print("\nAll experiments completed.")

Running experiment with seed 10...
Running experiment with seed 20...
Running experiment with seed 30...
Running experiment with seed 40...
Running experiment with seed 50...
Running experiment with seed 60...
Running experiment with seed 70...
Running experiment with seed 80...
Running experiment with seed 90...
Running experiment with seed 100...

All experiments completed.


In [9]:
results_df = pd.DataFrame(results)

print(results_df.round(4).to_string(index=False))

 Seed Channel               Model  Accuracy     Pd    Pfa     Pm  ROC_AUC  Threshold
   10    AWGN Logistic Regression    0.6776 0.4057 0.0505 0.5943   0.8390     0.7423
   10    AWGN       Random Forest    0.8271 0.6943 0.0400 0.3057   0.9418     0.6400
   20    AWGN Logistic Regression    0.6871 0.4286 0.0543 0.5714   0.8326     0.7229
   20    AWGN       Random Forest    0.8210 0.6829 0.0410 0.3171   0.9422     0.6350
   30    AWGN Logistic Regression    0.6662 0.3838 0.0514 0.6162   0.8338     0.7592
   30    AWGN       Random Forest    0.8262 0.6876 0.0352 0.3124   0.9413     0.6450
   40    AWGN Logistic Regression    0.6681 0.3810 0.0448 0.6190   0.8233     0.7618
   40    AWGN       Random Forest    0.8195 0.6867 0.0476 0.3133   0.9356     0.6400
   50    AWGN Logistic Regression    0.6743 0.3981 0.0495 0.6019   0.8282     0.7425
   50    AWGN       Random Forest    0.8271 0.7000 0.0457 0.3000   0.9413     0.6300
   60    AWGN Logistic Regression    0.6695 0.3895 0.0505 0.6105 

In [10]:
metrics = [
    "Accuracy",
    "Pd",
    "Pfa",
    "Pm",
    "ROC_AUC"
]

summary_df = results_df.groupby(
    ["Channel", "Model"]
)[metrics].agg(["mean", "std"])

print(summary_df.round(4))

                            Accuracy              Pd             Pfa          \
                                mean     std    mean     std    mean     std   
Channel Model                                                                  
AWGN    Logistic Regression   0.6721  0.0077  0.3932  0.0159  0.0490  0.0042   
        Random Forest         0.8240  0.0040  0.6934  0.0102  0.0453  0.0073   

                                 Pm         ROC_AUC          
                               mean     std    mean     std  
Channel Model                                                
AWGN    Logistic Regression  0.6068  0.0159   0.833  0.0058  
        Random Forest        0.3066  0.0102   0.940  0.0027  


In [11]:
# Save individual experiment results
results_df.to_csv(
    "statistical_validation_awgn_all_runs.csv",
    index=False
)

# Flatten the summary table for easier reporting
summary_export = results_df.groupby(
    ["Channel", "Model"]
)[metrics].agg(["mean", "std"])

summary_export.columns = [
    f"{metric}_{stat}"
    for metric, stat in summary_export.columns
]

summary_export = summary_export.reset_index()

summary_export.to_csv(
    "statistical_validation_awgn_summary.csv",
    index=False
)

print("Statistical validation results saved successfully.")

Statistical validation results saved successfully.


In [12]:
rayleigh_results = []

for seed in random_seeds:

    print(f"Running Rayleigh experiment with seed {seed}...")

    # Generate split
    train_idx, val_idx, test_idx = create_data_split(
        y, snr, seed
    )

    # Split Rayleigh features
    X_train = X_rayleigh_features[train_idx]
    X_val = X_rayleigh_features[val_idx]
    X_test = X_rayleigh_features[test_idx]

    y_train = y[train_idx]
    y_val = y[val_idx]
    y_test = y[test_idx]

    # --------------------------------
    # 1. LOGISTIC REGRESSION
    # --------------------------------

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    lr_model = LogisticRegression(
        max_iter=1000,
        random_state=seed,
        class_weight="balanced"
    )

    lr_model.fit(X_train_scaled, y_train)

    lr_val_prob = lr_model.predict_proba(X_val_scaled)[:, 1]
    lr_test_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

    lr_threshold = np.quantile(
        lr_val_prob[y_val == 0],
        0.95,
        method="higher"
    )

    lr_metrics = calculate_metrics(
        y_test, lr_test_prob, lr_threshold
    )

    rayleigh_results.append({
        "Seed": seed,
        "Channel": "Rayleigh",
        "Model": "Logistic Regression",
        **lr_metrics
    })

    # --------------------------------
    # 2. RANDOM FOREST
    # --------------------------------

    rf_model = RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=seed,
        n_jobs=-1,
        class_weight="balanced"
    )

    rf_model.fit(X_train, y_train)

    rf_val_prob = rf_model.predict_proba(X_val)[:, 1]
    rf_test_prob = rf_model.predict_proba(X_test)[:, 1]

    rf_threshold = np.quantile(
        rf_val_prob[y_val == 0],
        0.95,
        method="higher"
    )

    rf_metrics = calculate_metrics(
        y_test, rf_test_prob, rf_threshold
    )

    rayleigh_results.append({
        "Seed": seed,
        "Channel": "Rayleigh",
        "Model": "Random Forest",
        **rf_metrics
    })

print("\nAll Rayleigh experiments completed.")

Running Rayleigh experiment with seed 10...
Running Rayleigh experiment with seed 20...
Running Rayleigh experiment with seed 30...
Running Rayleigh experiment with seed 40...
Running Rayleigh experiment with seed 50...
Running Rayleigh experiment with seed 60...
Running Rayleigh experiment with seed 70...
Running Rayleigh experiment with seed 80...
Running Rayleigh experiment with seed 90...
Running Rayleigh experiment with seed 100...

All Rayleigh experiments completed.


In [13]:
rayleigh_results_df = pd.DataFrame(rayleigh_results)

print(
    rayleigh_results_df.round(4).to_string(index=False)
)

 Seed  Channel               Model  Accuracy     Pd    Pfa     Pm  ROC_AUC  Threshold
   10 Rayleigh Logistic Regression    0.6414 0.3371 0.0543 0.6629   0.7486     0.6998
   10 Rayleigh       Random Forest    0.7748 0.6105 0.0610 0.3895   0.8812     0.6250
   20 Rayleigh Logistic Regression    0.6524 0.3629 0.0581 0.6371   0.7542     0.6940
   20 Rayleigh       Random Forest    0.7767 0.6076 0.0543 0.3924   0.8862     0.6400
   30 Rayleigh Logistic Regression    0.6381 0.3390 0.0629 0.6610   0.7509     0.6813
   30 Rayleigh       Random Forest    0.7700 0.5771 0.0371 0.4229   0.8815     0.6650
   40 Rayleigh Logistic Regression    0.6310 0.3124 0.0505 0.6876   0.7566     0.7002
   40 Rayleigh       Random Forest    0.7752 0.5962 0.0457 0.4038   0.8806     0.6450
   50 Rayleigh Logistic Regression    0.6448 0.3343 0.0448 0.6657   0.7534     0.6956
   50 Rayleigh       Random Forest    0.7686 0.5838 0.0467 0.4162   0.8763     0.6550
   60 Rayleigh Logistic Regression    0.6452 0.3295 0.

In [14]:
rayleigh_summary_df = rayleigh_results_df.groupby(
    ["Channel", "Model"]
)[metrics].agg(["mean", "std"])

print(rayleigh_summary_df.round(4))

                             Accuracy              Pd             Pfa          \
                                 mean     std    mean     std    mean     std   
Channel  Model                                                                  
Rayleigh Logistic Regression   0.6418  0.0075  0.3339  0.0143  0.0504  0.0082   
         Random Forest         0.7723  0.0034  0.5945  0.0110  0.0499  0.0076   

                                  Pm         ROC_AUC          
                                mean     std    mean     std  
Channel  Model                                                
Rayleigh Logistic Regression  0.6661  0.0143  0.7550  0.0060  
         Random Forest        0.4055  0.0110  0.8819  0.0031  


In [15]:
# Combine individual experiment results
all_results_df = pd.concat(
    [results_df, rayleigh_results_df],
    ignore_index=True
)

# Save all individual runs
all_results_df.to_csv(
    "statistical_validation_all_runs.csv",
    index=False
)

# Calculate combined summary
combined_summary = all_results_df.groupby(
    ["Channel", "Model"]
)[metrics].agg(["mean", "std"])

# Flatten column names
combined_summary.columns = [
    f"{metric}_{stat}"
    for metric, stat in combined_summary.columns
]

combined_summary = combined_summary.reset_index()

# Save summary
combined_summary.to_csv(
    "statistical_validation_summary.csv",
    index=False
)

print("All statistical validation results saved.")
print("\nCombined summary:")
print(combined_summary.round(4).to_string(index=False))

All statistical validation results saved.

Combined summary:
 Channel               Model  Accuracy_mean  Accuracy_std  Pd_mean  Pd_std  Pfa_mean  Pfa_std  Pm_mean  Pm_std  ROC_AUC_mean  ROC_AUC_std
    AWGN Logistic Regression         0.6721        0.0077   0.3932  0.0159    0.0490   0.0042   0.6068  0.0159        0.8330       0.0058
    AWGN       Random Forest         0.8240        0.0040   0.6934  0.0102    0.0453   0.0073   0.3066  0.0102        0.9400       0.0027
Rayleigh Logistic Regression         0.6418        0.0075   0.3339  0.0143    0.0504   0.0082   0.6661  0.0143        0.7550       0.0060
Rayleigh       Random Forest         0.7723        0.0034   0.5945  0.0110    0.0499   0.0076   0.4055  0.0110        0.8819       0.0031
